In [4]:
import matplotlib.pyplot as plt
import networkx as nx
import seaborn as sns
import torch as th
import torch.nn.functional as F
from matplotlib.cm import ScalarMappable
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch_geometric.nn import VGAE, GCNConv
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.utils import from_networkx, to_networkx
import numpy as np
import pandas as pd

In [ ]:
#Lecture avec NetworkX
G = nx.read_graphml("airportsAndCoordAndPop.graphml")

# On met toutes les caractéristiques dans un DF
data_list = []
for node_id, attrs in G.nodes(data=True):
    data_list.append({
        "id": node_id,
        "city_name": attrs.get("city_name"),
        "country": attrs.get("country"),
        "lat": float(attrs.get("lat", 0.0)),
        "lon": float(attrs.get("lon", 0.0)),
        "population": int(attrs.get("population", 0)),
    })

df = pd.DataFrame(data_list)

#Aperçu rapide
print("\n Aperçu des données(10ere lignes)")
print(df.head(10))
#Liaisons => arêtes Noeuds => aéroports


 Aperçu des données
  id       city_name           country        lat         lon  population
0  0            Anaa  FRENCH_POLYNESIA -17.353889 -145.509722       10000
1  1      Hao Island  FRENCH_POLYNESIA -18.066667 -140.950000       10000
2  2         Papeete  FRENCH_POLYNESIA -17.550000 -149.600000       26357
3  3  Gambier Island  FRENCH_POLYNESIA -23.033333 -135.000000       10000
4  4          Makemo  FRENCH_POLYNESIA -16.584889 -143.657250       10000
5  5        Auckland       NEW_ZEALAND -37.008056  174.791667      417910
6  6          Atuona  FRENCH_POLYNESIA  -9.800000 -139.033333       10000
7  7       Bora Bora  FRENCH_POLYNESIA -16.450000 -151.750000       10000
8  8        Honolulu               USA  21.318691 -157.922407      371657
9  9         Huahine  FRENCH_POLYNESIA -16.680000 -151.000000       10000


In [ ]:
# S'assurer que les identifiants de nœuds sont bien des entiers contigus 0..N-1
G = nx.convert_node_labels_to_integers(G)

print(f"Nombre d'aéroports : {G.number_of_nodes()}")
print(f"Nombre de liaisons  : {G.number_of_edges()}")

# nom des villes et pays
cities = [G.nodes[i].get("city_name", f"Airport_{i}") for i in range(len(G))]
countries = [G.nodes[i].get("country", "Unknown") for i in range(len(G))]

# 3. Extraction des attributs d'entrée X (lat, lon) et de la cible Y (population)
lat = np.array([float(G.nodes[i]["lat"]) for i in range(len(G))])
lon = np.array([float(G.nodes[i]["lon"]) for i in range(len(G))])
pop = np.array([float(G.nodes[i]["population"]) for i in range(len(G))])

# 4. Conversion vers l'objet Data de PyG
data = from_networkx(G)

Nombre de nœuds (aéroports) : 3363
Nombre d'arêtes (liaisons)  : 13547


In [3]:
# Normalisation des coordonnées spatiales
lat_norm = (lat - lat.mean()) / lat.std()
lon_norm = (lon - lon.mean()) / lon.std()
x = np.column_stack([lat_norm, lon_norm])
data.x = th.tensor(x, dtype=th.float)

# Cible en échelle logarithmique log10
log_pop = np.log10(np.maximum(pop, 1.0))
data.y = th.tensor(log_pop, dtype=th.float).unsqueeze(1)

# Définition des masques :
# - Les nœuds avec pop == 10000 sont suspects et réservés au test d'anomalies
# - Parmi les nœuds plausibles (pop != 10000), 80 % servent à l'entraînement
is_default_pop = pop == 10000.0
plausible_indices = np.where(~is_default_pop)[0]

num_train = int(0.8 * len(plausible_indices))
np.random.shuffle(plausible_indices)
train_idx = plausible_indices[:num_train]

train_mask = th.zeros(data.num_nodes, dtype=th.bool)
train_mask[train_idx] = True

data.train_mask = train_mask
data.test_mask = ~train_mask  # Contient les 20% de validation + tous les nœuds suspects (10 000)

print(f"Nœuds d'entraînement : {data.train_mask.sum().item()}")
print(f"Nœuds à évaluer     : {data.test_mask.sum().item()} (dont {is_default_pop.sum()} avec pop = 10000)")

Nœuds d'entraînement : 1352
Nœuds à évaluer     : 2011 (dont 1673 avec pop = 10000)
